In [29]:
import datetime 
from dateutil.relativedelta import relativedelta
import pandas as pd
import numpy as np
import db_dtypes
import pymssql
from shutil import copyfile
from openpyxl import load_workbook
import os
from google.cloud import bigquery
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = 'BQ.json'

DB_info = {'server':'192.168.61.119:7622', 'user':'BAReporting', 'password':'KeHeCReme8he'}

client = bigquery.Client()

drop reason:

or.takeaway.order: 'quit_order',                  # 瀏覽菜單
or.takeaway.checkout: 'quit_checkout',            # 選擇優惠及下單
or.takeaway.place-order: 'quit_place-order',      # Rice dollar + 繼續

or.takeaway.payment

In [30]:
sql = """
WITH drop_sessions AS (
  SELECT
    DeviceId,
    SessionId,
  FROM `openrice-production.ORGA.PV_20260915`
  WHERE DeviceId IS NOT NULL
    AND SessionId IS NOT NULL
  GROUP BY DeviceId, SessionId
  HAVING
    COUNTIF(LOWER(TRIM(EventAction)) = 'or.takeaway.order') > 0         -- must hv order event
    AND COUNTIF(                                                -- session hv no pay event
      LOWER(TRIM(EventAction)) = 'or.takeaway.pay'
      OR LOWER(COALESCE(EventLabelRaw, '')) LIKE '%or.takeaway.pay%'
    ) = 0
),

sampled_drop_sessions AS (
  SELECT
    DeviceId,
    SessionId
  FROM drop_sessions
  ORDER BY RAND()
  LIMIT 2000
)

SELECT pv.SessionId, pv.DeviceId, pv.Time, pv.EventAction, pv.EventLabelRaw
FROM `openrice-production.ORGA.PV_20260915` AS pv
INNER JOIN sampled_drop_sessions AS ds
  ON pv.DeviceId = ds.DeviceId
  AND pv.SessionId = ds.SessionId
ORDER BY pv.DeviceId, pv.SessionId, pv.Time;
"""

df_bq = client.query(sql).result().to_dataframe()
df_bq

C:\Users\lenalee\AppData\Roaming\Python\Python313\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,SessionId,DeviceId,Time,EventAction,EventLabelRaw
0,129491,,2026-09-15 00:00:01.500000+00:00,or.poi.feature,CityID:704;userIP:66.249.74.197
1,129491,,2026-09-15 00:00:02.400000+00:00,impression.poi,CityID:9;PoiId:9561144
2,129491,,2026-09-15 00:00:04.400000+00:00,or.poi.feature,CityID:704;userIP:66.249.74.193
3,129491,,2026-09-15 00:00:05.500000+00:00,,https://www.openrice.com/zh/foshan/r-貴州黃記牛肉館-勒...
4,129491,,2026-09-15 00:00:07.800000+00:00,or.poi.feature,CityID:704;userIP:66.249.74.193
...,...,...,...,...,...
377709,1030289,ffd37b15-5d2b-4dab-b9f0-09d7ec0e1ecb,2026-09-15 22:24:54.400000+00:00,impression.ad.1,CityID:0;Lang:hk;appVersion:7.20.5;sn:HK.Searc...
377710,1030289,ffd37b15-5d2b-4dab-b9f0-09d7ec0e1ecb,2026-09-15 22:24:54.400000+00:00,impression.ad.1,CityID:0;Lang:hk;appVersion:7.20.5;sn:HK.Searc...
377711,1030289,ffd37b15-5d2b-4dab-b9f0-09d7ec0e1ecb,2026-09-15 22:24:54.400000+00:00,impression.ad.1,CityID:0;Lang:hk;appVersion:7.20.5;sn:HK.Searc...
377712,1030289,ffd37b15-5d2b-4dab-b9f0-09d7ec0e1ecb,2026-09-15 22:25:00.900000+00:00,or.sr1.reel.photo,CityID:0;Lang:hk;Ver:7.20.5;tracking:ekeywords...


In [ ]:
#df_bq.to_csv("session.csv", index=False, encoding="utf-8-sig")


In [31]:
# 按時間倒序，get session last quit point
tracking_source = df_bq.copy()
tracking_source["Time"] = pd.to_datetime(tracking_source["Time"])
tracking_source["normalized_action"] = (
    tracking_source["EventAction"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

tracking_source = tracking_source.sort_values(
    ["DeviceId", "SessionId", "Time"],
    kind="stable",
)

quit_point_mapping = {
    "or.takeaway.order": "quit_order",
    "or.takeaway.checkout": "quit_checkout",
    "or.takeaway.place-order": "quit_place-order",
}

def classify_quit_point(actions):
    for action in reversed(actions.tolist()):
        if action in quit_point_mapping:
            return quit_point_mapping[action]
    return pd.NA

drop_session_summary = (
    tracking_source
    .groupby(["SessionId", "DeviceId"], sort=False)["normalized_action"]
    .agg(classify_quit_point)
    .rename("quit_point")
    .reset_index()
)

drop_session_summary

,SessionId,DeviceId,quit_point
0,129491,,quit_order
1,618,002946660a761360,quit_order
2,1054,00457015-053d-4d70-9146-ff4e9cb92198,quit_order
3,1396,005b47f7-dd47-4aac-a4d9-a1e93bd0bdb1,quit_order
4,1986,0081b825-a7bc-4709-a152-af69ad687ceb,quit_order
...,...,...,...
1995,1027438,ff200382-ebc8-4ef0-8308-6970a75fe1fa,quit_order
1996,1029007,ff8346d9-1872-4502-91af-8c5ae9852702,quit_order
1997,1029109,ff8973697fe7ca96,quit_order
1998,1030168,ffccbfbb-85ac-44f5-af9b-c3d02406b8c6,quit_order


In [32]:
# quit point count and %
quit_point_result = (
    drop_session_summary["quit_point"]
    .value_counts()
    .reset_index(name="count")
)

quit_point_result["%"] = (
    quit_point_result["count"]
    .div(quit_point_result["count"].sum())
    .mul(100)
    .round(2)
)

quit_point_result

,quit_point,count,%
0,quit_order,1946,97.3
1,quit_checkout,38,1.9
2,quit_place-order,16,0.8
